[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_77_Phase9_MultiAgent_Orchestration.ipynb)

# Lesson 77 — Phase 9 Kickoff: Advanced Multi-Agent Orchestration

**Where you are:** You've spent 76 lessons and eight phases building *one agent at a time*. Phase 6 shipped **paper-distiller** (an agent). Phase 7 shipped **agent-bench** (offline eval). Phase 8 shipped **agent-obs** (production ops). Every one of those is a *single* agent, made reliable.

**Phase 9 changes the unit of design.** The question stops being *"how do I make this agent good?"* and becomes *"how do I make several agents work together as a system?"* — a supervisor that delegates, workers that specialize, a critic that checks, a router that dispatches. This is **orchestration**, and it is where most real production LLM systems actually live.

> **The one idea for today:** a multi-agent system is not "more AI." It is a **control-flow topology** over LLM calls — who talks to whom, in what order, and who decides when it's done. The intelligence is the same model you already have; the leverage is the *wiring*.

### Phase 9 roadmap (track 9A — Advanced Multi-Agent Orchestration)

| Lesson | Topic | The question it answers |
|---|---|---|
| **77 (today)** | **Orchestration topologies** | What are the shapes, and when does each pay off? |
| 78 | Shared state & the blackboard | How do agents coordinate without drowning in messages? |
| 79 | Routing & handoff | How does a request find the right specialist? |
| 80 | Planning & task decomposition | How does a supervisor break a goal into a DAG of subtasks? |
| 81 | Reliability: retries, timeouts, partial failure | What happens when one agent in the mesh dies? |
| 82 | Phase 9 capstone | Ship a small orchestration framework / app as a 4th OSS artifact |

*(Roadmap is a hypothesis — it will adapt to your questions and interests, but it stays incremental.)*

Everything below runs **with no API key** — the agent "brains" are deterministic mocks so you can watch the *orchestration* logic without spending a cent or waiting on the network. Each mock has an optional real-Claude path gated on `HAVE_API_KEY`, exactly like earlier lessons, so the structure transfers verbatim to production.

## 1. One agent vs. a system of agents

You already know how to make a single agent loop: prompt → tool call → observe → repeat. That is a **single-agent** system, and it should be your default. Reaching for multiple agents adds real cost, so you need to know *why* you're doing it.

| | **Single agent** | **Multi-agent system** |
|---|---|---|
| Unit of design | one prompt + tool loop | a **topology** of agents + messages |
| Strength | simple, cheap, low latency | specialization, parallelism, checking |
| Failure mode | one blind spot, no second opinion | coordination overhead, error propagation |
| Cost | 1× tokens | 2–5× tokens (more calls) |
| Latency | one chain | can be *lower* (parallel) or *higher* (chains) |
| When to reach for it | almost always, first | when a single agent measurably plateaus |

### When multi-agent actually helps (and when it doesn't)

**Helps when:**
- The task **decomposes** into independent sub-tasks (research 5 sub-questions in parallel).
- You benefit from a **second opinion** — a generator makes mistakes a *verifier* reliably catches (the generator↔verifier asymmetry we'll measure today).
- Different steps need different **tools, context, or personas** (a coder agent vs. a security-reviewer agent).

**Hurts when:**
- The task is a single tight reasoning chain — splitting it just adds hand-off noise.
- You add agents hoping "more AI = better." Every hop is another chance to lose information and another bill.

> Rule of thumb: **start with one agent, measure where it plateaus, and add exactly the agent that closes that specific gap.** Today we'll *measure* one such plateau and close it.

In [ ]:
# === Setup — deterministic, no API key required ===
# Colab: this installs `rich` for pretty tables. Nothing here calls the network.
try:
    from rich import print as rprint
    from rich.table import Table
    from rich.console import Console
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rich"], check=False)
    from rich import print as rprint
    from rich.table import Table
    from rich.console import Console

import os, time, random, base64, textwrap
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable
from concurrent.futures import ThreadPoolExecutor

console = Console()
random.seed(77)                     # fully reproducible
BASE = "/content"                   # Colab working dir
Path(BASE).mkdir(parents=True, exist_ok=True)

# Optional real-LLM path — OFF by default. The whole lesson runs without it.
HAVE_API_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
rprint(f"[bold green]Setup ready.[/] HAVE_API_KEY={HAVE_API_KEY} (deterministic mocks used either way)")

## 2. The substrate: messages and agents

Before we can wire agents together we need two primitives, and only two:

1. **`Message`** — the envelope agents pass around. Sender, recipient, a `kind` (`task` / `result` / `critique` / `revision`), the payload, and metadata (cost, latency). *Everything* in an orchestration is messages moving between agents — if you can log the messages, you can debug the system.
2. **`Agent`** — a named role with a `backend` (its "brain"). `act(msg)` takes a message in and returns a message out. In production the backend is an LLM call; here it's a deterministic Python function so the *orchestration* is what we study.

This is deliberately tiny. A multi-agent framework is not a big idea — it's these two types plus disciplined control flow.

In [ ]:
# === Message + Agent primitives ===
@dataclass
class Message:
    sender: str
    recipient: str
    kind: str                 # "task" | "result" | "critique" | "revision"
    content: Any
    meta: dict = field(default_factory=dict)

# A "backend" is a callable: (agent_name, input_content) -> (output_content, tokens)
# In prod this wraps an LLM. Here it's deterministic so the demo never flakes.
class Agent:
    def __init__(self, name: str, role: str, backend: Callable):
        self.name = name
        self.role = role
        self.backend = backend
        self.calls = 0                       # how many times this agent "thought"
    def act(self, msg: Message) -> Message:
        self.calls += 1
        out, tokens = self.backend(self.name, msg.content)
        return Message(sender=self.name, recipient=msg.sender,
                       kind="result", content=out,
                       meta={"tokens": tokens})

# --- self-test: an echo agent round-trips a message ---
echo = Agent("echo", "test", backend=lambda name, c: (f"echo:{c}", 3))
_r = echo.act(Message("user", "echo", "task", "hello"))
assert _r.content == "echo:hello" and _r.meta["tokens"] == 3 and echo.calls == 1
rprint("[green]OK[/] Message/Agent primitives round-trip. One agent = one callable brain behind a name.")

## 3. A task with ground truth (so we can *measure* topologies)

Slogans about multi-agent systems are cheap; we're going to **measure**. We need a task suite where every item has a known correct answer, and a "solver" agent that is *good but not perfect* — exactly the situation where orchestration earns its keep.

**The task:** compute an invoice total. Each task is a list of line items `(qty, unit_price, discount_pct)` plus a `tax_rate`. The correct rule is: **apply the per-item discount first, then tax the discounted subtotal.**

**The solver's flaw (a real one):** on "hard" invoices — those with *both* a discount *and* a nonzero tax — our solver applies tax to the *pre-discount* subtotal and subtracts the discount afterward. That is a genuine, common ordering bug. On easy invoices (no discount, or no tax) the two orders coincide and the solver is correct.

This gives us a generator that's right about half the time — a plateau we can see and then close.

In [ ]:
# === Ground-truth task suite + a flawed solver ===
@dataclass
class Task:
    id: int
    items: list          # list of (qty, unit_price, discount_pct)
    tax_rate: float

def is_hard(t: Task) -> bool:
    return any(d > 0 for _, _, d in t.items) and t.tax_rate > 0

def true_total(t: Task) -> float:
    subtotal = sum(q * p * (1 - d / 100) for q, p, d in t.items)   # discount THEN tax
    return round(subtotal * (1 + t.tax_rate), 2)

def make_tasks(n=40) -> list:
    tasks = []
    for i in range(n):
        k = random.randint(1, 3)
        items = [(random.randint(1, 5),
                  round(random.uniform(5, 50), 2),
                  random.choice([0, 0, 10, 20, 25])) for _ in range(k)]
        tax = random.choice([0.0, 0.0, 0.05, 0.08, 0.10])
        tasks.append(Task(i, items, tax))
    return tasks

# ---- the solver "brain": correct on easy tasks, ordering-buggy on hard ones ----
def solver_backend(name, task: Task):
    if not is_hard(task):
        return true_total(task), 40                      # easy: gets it right
    # BUG: tax the pre-discount subtotal, then subtract discount at the end
    sub_nodisc = sum(q * p for q, p, _ in task.items)
    taxed = sub_nodisc * (1 + task.tax_rate)
    disc = sum(q * p * (d / 100) for q, p, d in task.items)
    return round(taxed - disc, 2), 40                    # hard: wrong order

TASKS = make_tasks(40)
solver = Agent("solver", "compute invoice total", solver_backend)

def accuracy(answers, tasks, eps=0.01):
    hits = sum(1 for a, t in zip(answers, tasks) if abs(a - true_total(t)) < eps)
    return hits / len(tasks)

# --- Topology 0: SINGLE AGENT baseline ---
solo_answers = [solver.act(Message("user", "solver", "task", t)).content for t in TASKS]
BASE_ACC = accuracy(solo_answers, TASKS)
n_hard = sum(is_hard(t) for t in TASKS)
rprint(f"[bold]Single-agent baseline:[/] accuracy = [yellow]{BASE_ACC:.0%}[/] "
       f"({n_hard}/{len(TASKS)} tasks are 'hard' and expose the ordering bug)")
assert 0.4 <= BASE_ACC <= 0.75, BASE_ACC   # a real, visible plateau

## 4. Topology A — Orchestrator ↔ Worker (the workhorse)

The single most useful pattern in production: a **supervisor** (a.k.a. orchestrator/manager) that owns the goal, **decomposes** it into sub-tasks, **dispatches** each to a specialist **worker**, and **aggregates** the results. The workers never talk to each other — they only talk to the boss. This "star" shape keeps coordination simple and is how tools like research assistants and coding agents are structured.

We'll show it on a *decomposable* task — writing a short research brief — because decomposition is what this topology is for. The workers pull from a tiny deterministic knowledge base so the demo is reproducible; swap the backend for an LLM+RAG and the wiring is unchanged.

In [ ]:
# === Topology A: supervisor decomposes -> workers answer -> supervisor writes ===
KB = {
    "caching":  "Cache identical prompts to cut cost and p95 latency.",
    "batching": "Batch independent requests to raise throughput.",
    "streaming":"Stream tokens so users see progress before completion.",
    "eval":     "Gate every deploy behind an offline eval suite.",
}

def worker_backend(name, subtopic):
    return KB.get(subtopic, "no data on file"), 25

def supervisor_decompose(goal: str) -> list:
    # A real supervisor would ask an LLM to plan; here we map goal -> subtopics.
    return [w for w in KB if w in goal] or list(KB)[:2]

def orchestrate(goal: str, workers: dict):
    plan = supervisor_decompose(goal)                       # 1. decompose
    trace, findings = [], {}
    for sub in plan:                                        # 2. dispatch
        res = workers[sub].act(Message("supervisor", sub, "task", sub))
        findings[sub] = res.content
        trace.append((sub, res.content, res.meta["tokens"]))
    brief = " ".join(f"[{k}] {v}" for k, v in findings.items())  # 3. aggregate
    return brief, plan, trace

workers = {sub: Agent(sub, "researcher", worker_backend) for sub in KB}
goal = "Write a brief on caching, batching and eval for LLM apps."
brief, plan, trace = orchestrate(goal, workers)

tbl = Table(title="Supervisor -> Worker trace", show_lines=False)
tbl.add_column("worker"); tbl.add_column("finding"); tbl.add_column("tok", justify="right")
for sub, content, tok in trace:
    tbl.add_row(sub, content, str(tok))
console.print(tbl)
rprint(f"[bold]Aggregated brief:[/] {brief}")

assert set(plan) == {"caching", "batching", "eval"}          # decomposed correctly
assert all(sub in brief for sub in plan)                     # every worker contributed
rprint("[green]OK[/] Orchestrator-worker: one boss, N specialists, results merged. Workers never talk to each other.")

## 5. Topology B & C — Sequential pipeline vs. Parallel fan-out

Two more shapes, and the difference between them is **latency**, not intelligence:

- **Sequential pipeline (chain):** agent 1 → agent 2 → agent 3, each consuming the previous output. Use it when steps *depend* on each other (extract → summarize → translate). Wall-clock ≈ **sum** of step latencies.
- **Parallel fan-out / fan-in (map-reduce):** dispatch N *independent* sub-tasks at once, then merge. Use it when sub-tasks *don't* depend on each other. Wall-clock ≈ **max** of the step latencies, not the sum.

The orchestrator-worker task above has independent sub-tasks, so it's a natural fan-out. Let's prove the latency win with real wall-clock time (each worker "thinks" for a simulated 60 ms).

In [ ]:
# === Topology B (sequential) vs C (parallel) — same work, different wall-clock ===
LAT = 0.06   # simulated per-agent "thinking" latency, seconds

def slow_worker(name, subtopic):
    time.sleep(LAT)
    return KB.get(subtopic, "no data"), 25

subs = list(KB)                      # 4 independent sub-tasks
slow = {s: Agent(s, "researcher", slow_worker) for s in subs}

# Sequential: one after another -> time ~ sum
t0 = time.perf_counter()
seq = [slow[s].act(Message("sup", s, "task", s)).content for s in subs]
seq_time = time.perf_counter() - t0

# Parallel fan-out: all at once via a thread pool -> time ~ max
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=len(subs)) as pool:
    par = list(pool.map(lambda s: slow[s].act(Message("sup", s, "task", s)).content, subs))
par_time = time.perf_counter() - t0

rprint(f"Sequential: [yellow]{seq_time*1000:5.0f} ms[/]  (~sum of {len(subs)} x {LAT*1000:.0f}ms)")
rprint(f"Parallel:   [green]{par_time*1000:5.0f} ms[/]  (~max of the {len(subs)} calls)")
rprint(f"Speed-up:   [bold]{seq_time/par_time:.1f}x[/]")

assert seq == par                                  # identical results...
assert par_time < seq_time * 0.6                   # ...but parallel is much faster
rprint("[green]OK[/] Independent sub-tasks -> fan out. Dependent steps -> chain. The topology sets your latency.")

## 6. Topology D — Evaluator ↔ Optimizer (the reflection / critic loop)

This is the pattern that closes the plateau we measured in section 3. It rests on a real, exploited asymmetry:

> **Generating a correct answer is harder than checking one.** A second agent whose *only job* is to verify can catch mistakes the generator made — even when it's the same underlying model — because checking is an easier problem than producing.

The loop is: **generate → critique → (if flawed) revise → repeat**. Our critic recomputes the invoice with the *correct* operation order (an independent method) and flags disagreements; the reviser adopts the corrected value. In production, the critic is a second LLM instance with a verification-focused prompt — the *structure* is what matters, and it's identical.

Let's run the exact same task suite through the loop and measure the lift over the single-agent baseline.

In [ ]:
# === Topology D: generate -> critique -> revise, and MEASURE the lift ===
def critic_backend(name, payload):
    # payload = (task, candidate_answer). Independent re-derivation = the check.
    task, cand = payload
    correct = true_total(task)
    ok = abs(cand - correct) < 0.01
    critique = "looks correct" if ok else "ordering error: discount before tax"
    return (ok, correct, critique), 30

critic = Agent("critic", "verify answer", critic_backend)

def reflect_solve(task, solver, critic, max_rounds=2):
    msg = solver.act(Message("user", "solver", "task", task))
    ans = msg.content
    for _ in range(max_rounds):
        crit = critic.act(Message("solver", "critic", "critique", (task, ans)))
        ok, suggested, note = crit.content
        if ok:
            break
        ans = suggested                      # reviser adopts the verified value
    return ans

# reset call counters for a clean cost comparison
solver.calls = critic.calls = 0
multi_answers = [reflect_solve(t, solver, critic) for t in TASKS]
MULTI_ACC = accuracy(multi_answers, TASKS)

tbl = Table(title="Single agent vs. critic loop (same 40 tasks)")
tbl.add_column("topology"); tbl.add_column("accuracy", justify="right")
tbl.add_row("Single agent (solver only)", f"{BASE_ACC:.0%}")
tbl.add_row("Multi-agent (solver + critic loop)", f"{MULTI_ACC:.0%}")
console.print(tbl)

assert MULTI_ACC > BASE_ACC + 0.20        # a large, real lift
assert MULTI_ACC >= 0.95                  # critic closes the ordering-bug plateau
rprint(f"[bold green]PAYOFF:[/] {BASE_ACC:.0%} -> {MULTI_ACC:.0%}. Same model, same tasks — the [italic]second opinion[/] did it.")

## 7. The bill: multi-agent buys quality with tokens and latency

The critic loop didn't get smarter for free. Every hop is another model call. Before you reach for orchestration, you must be able to state the trade you're making: **more agents = more tokens + (often) more latency, in exchange for quality or parallelism.**

This is exactly the discipline from Phase 8 — you already know how to *measure* cost and latency with agent-obs. Now measure the cost of the *topology itself*.

In [ ]:
# === Count the cost of the topology ===
# Rebuild both runs with fresh counters and sum tokens.
def run_single(tasks, solver):
    solver.calls = 0; toks = 0
    for t in tasks:
        m = solver.act(Message("u", "solver", "task", t)); toks += m.meta["tokens"]
    return solver.calls, toks

def run_multi(tasks, solver, critic):
    solver.calls = critic.calls = 0; toks = 0
    for t in tasks:
        m = solver.act(Message("u", "solver", "task", t)); toks += m.meta["tokens"]
        for _ in range(2):
            c = critic.act(Message("solver", "critic", "critique", (t, m.content)))
            toks += c.meta["tokens"]
            ok, suggested, _ = c.content
            if ok: break
            m = Message("solver", "u", "result", suggested, {"tokens": 0})
    return solver.calls + critic.calls, toks

s_calls, s_tok = run_single(TASKS, solver)
m_calls, m_tok = run_multi(TASKS, solver, critic)

tbl = Table(title="Cost of quality")
for c in ("topology", "model calls", "tokens", "accuracy"):
    tbl.add_column(c, justify="right" if c != "topology" else "left")
tbl.add_row("Single agent",  str(s_calls), str(s_tok), f"{BASE_ACC:.0%}")
tbl.add_row("Critic loop",   str(m_calls), str(m_tok), f"{MULTI_ACC:.0%}")
console.print(tbl)

rprint(f"The critic loop used [yellow]{m_tok/s_tok:.1f}x the tokens[/] to buy "
       f"[green]+{(MULTI_ACC-BASE_ACC):.0%} accuracy[/].")
assert m_calls > s_calls and m_tok > s_tok        # orchestration is never free
rprint("[green]OK[/] Always know the trade: reach for the extra agent only when the quality/parallelism is worth the bill.")

In [ ]:
# === Package the four topologies into a reusable module (orchestra/core.py) ===
# The source is embedded as base64 so no quote/docstring can ever collide with the cell.
import base64
from pathlib import Path
_B64 = "IiIib3JjaGVzdHJhLmNvcmUg4oCUIG1pbmltYWwgbXVsdGktYWdlbnQgb3JjaGVzdHJhdGlvbiBwcmltaXRpdmVzLgoKRm91ciB0b3BvbG9naWVzLCBvbmUgc3Vic3RyYXRlLiBTd2FwIHRoZSBkZXRlcm1pbmlzdGljIGJhY2tlbmRzIGZvciBMTE0gY2FsbHMKYW5kIHRoZSBjb250cm9sIGZsb3cgaXMgdW5jaGFuZ2VkLgoiIiIKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBDYWxsYWJsZQpmcm9tIGNvbmN1cnJlbnQuZnV0dXJlcyBpbXBvcnQgVGhyZWFkUG9vbEV4ZWN1dG9yCgoKQGRhdGFjbGFzcwpjbGFzcyBNZXNzYWdlOgogICAgc2VuZGVyOiBzdHIKICAgIHJlY2lwaWVudDogc3RyCiAgICBraW5kOiBzdHIgICAgICAgICAgICAgICAgICAgICAjICJ0YXNrIiB8ICJyZXN1bHQiIHwgImNyaXRpcXVlIiB8ICJyZXZpc2lvbiIKICAgIGNvbnRlbnQ6IEFueQogICAgbWV0YTogZGljdCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQoKCmNsYXNzIEFnZW50OgogICAgIiIiQSBuYW1lZCByb2xlIHdyYXBwaW5nIGEgYmFja2VuZCBjYWxsYWJsZTogKG5hbWUsIGNvbnRlbnQpIC0+IChvdXQsIHRva2VucykuIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgbmFtZTogc3RyLCByb2xlOiBzdHIsIGJhY2tlbmQ6IENhbGxhYmxlKToKICAgICAgICBzZWxmLm5hbWUsIHNlbGYucm9sZSwgc2VsZi5iYWNrZW5kLCBzZWxmLmNhbGxzID0gbmFtZSwgcm9sZSwgYmFja2VuZCwgMAoKICAgIGRlZiBhY3Qoc2VsZiwgbXNnOiAiTWVzc2FnZSIpIC0+ICJNZXNzYWdlIjoKICAgICAgICBzZWxmLmNhbGxzICs9IDEKICAgICAgICBvdXQsIHRva2VucyA9IHNlbGYuYmFja2VuZChzZWxmLm5hbWUsIG1zZy5jb250ZW50KQogICAgICAgIHJldHVybiBNZXNzYWdlKHNlbGYubmFtZSwgbXNnLnNlbmRlciwgInJlc3VsdCIsIG91dCwgeyJ0b2tlbnMiOiB0b2tlbnN9KQoKCmRlZiBzZXF1ZW50aWFsKGFnZW50cywgaW5pdGlhbCk6CiAgICAiIiJUb3BvbG9neSBCOiBjaGFpbiBhZ2VudHMsIGVhY2ggY29uc3VtaW5nIHRoZSBwcmV2aW91cyBvdXRwdXQuIiIiCiAgICBjb250ZW50ID0gaW5pdGlhbAogICAgZm9yIGEgaW4gYWdlbnRzOgogICAgICAgIGNvbnRlbnQgPSBhLmFjdChNZXNzYWdlKCJwaXBlIiwgYS5uYW1lLCAidGFzayIsIGNvbnRlbnQpKS5jb250ZW50CiAgICByZXR1cm4gY29udGVudAoKCmRlZiBwYXJhbGxlbChhZ2VudF9tYXAsIHN1YnRhc2tzLCBtYXhfd29ya2Vycz1Ob25lKToKICAgICIiIlRvcG9sb2d5IEM6IGZhbiBvdXQgaW5kZXBlbmRlbnQgc3VidGFza3MsIGZhbiBpbiByZXN1bHRzIChkaWN0KS4iIiIKICAgIG13ID0gbWF4X3dvcmtlcnMgb3IgbWF4KDEsIGxlbihzdWJ0YXNrcykpCiAgICBkZWYgX3J1bihzKToKICAgICAgICByZXR1cm4gcywgYWdlbnRfbWFwW3NdLmFjdChNZXNzYWdlKCJzdXAiLCBzLCAidGFzayIsIHMpKS5jb250ZW50CiAgICB3aXRoIFRocmVhZFBvb2xFeGVjdXRvcihtYXhfd29ya2Vycz1tdykgYXMgcG9vbDoKICAgICAgICByZXR1cm4gZGljdChwb29sLm1hcChfcnVuLCBzdWJ0YXNrcykpCgoKZGVmIG9yY2hlc3RyYXRlKGdvYWwsIGRlY29tcG9zZSwgd29ya2VycywgYWdncmVnYXRlPU5vbmUpOgogICAgIiIiVG9wb2xvZ3kgQTogc3VwZXJ2aXNvciBkZWNvbXBvc2VzIC0+IHdvcmtlcnMgLT4gYWdncmVnYXRlLiIiIgogICAgcGxhbiA9IGRlY29tcG9zZShnb2FsKQogICAgZmluZGluZ3MgPSBwYXJhbGxlbCh3b3JrZXJzLCBwbGFuKQogICAgaWYgYWdncmVnYXRlIGlzIE5vbmU6CiAgICAgICAgYWdncmVnYXRlID0gbGFtYmRhIGcsIGY6ICIgIi5qb2luKGYiW3trfV0ge3Z9IiBmb3IgaywgdiBpbiBmLml0ZW1zKCkpCiAgICByZXR1cm4gYWdncmVnYXRlKGdvYWwsIGZpbmRpbmdzKSwgcGxhbgoKCmRlZiByZWZsZWN0KHRhc2ssIHNvbHZlciwgY3JpdGljLCBtYXhfcm91bmRzPTIpOgogICAgIiIiVG9wb2xvZ3kgRDogZ2VuZXJhdGUgLT4gY3JpdGlxdWUgLT4gcmV2aXNlLiBjcml0aWMgYmFja2VuZCByZXR1cm5zCiAgICAob2s6IGJvb2wsIHN1Z2dlc3RlZCwgbm90ZSkuIFJldHVybnMgdGhlIGZpbmFsIGFuc3dlci4iIiIKICAgIGFucyA9IHNvbHZlci5hY3QoTWVzc2FnZSgidXNlciIsIHNvbHZlci5uYW1lLCAidGFzayIsIHRhc2spKS5jb250ZW50CiAgICBmb3IgXyBpbiByYW5nZShtYXhfcm91bmRzKToKICAgICAgICBvaywgc3VnZ2VzdGVkLCBfbm90ZSA9IGNyaXRpYy5hY3QoCiAgICAgICAgICAgIE1lc3NhZ2Uoc29sdmVyLm5hbWUsIGNyaXRpYy5uYW1lLCAiY3JpdGlxdWUiLCAodGFzaywgYW5zKSkpLmNvbnRlbnQKICAgICAgICBpZiBvazoKICAgICAgICAgICAgYnJlYWsKICAgICAgICBhbnMgPSBzdWdnZXN0ZWQKICAgIHJldHVybiBhbnMK"
Path(BASE, "orchestra").mkdir(parents=True, exist_ok=True)
Path(BASE, "orchestra", "__init__.py").write_text(
    "from .core import Message, Agent, sequential, parallel, orchestrate, reflect\n")
Path(BASE, "orchestra", "core.py").write_text(base64.b64decode(_B64).decode())

# import it back and smoke-test all four topologies
import sys
sys.path.insert(0, BASE)
import importlib, orchestra
importlib.reload(orchestra)
from orchestra import Message as M2, Agent as A2, sequential, parallel, orchestrate, reflect

up   = A2("up",  "x", lambda n, c: (str(c).upper(), 1))
bang = A2("bang","x", lambda n, c: (str(c) + "!", 1))
assert sequential([up, bang], "hi") == "HI!"                      # chain
kb = {"a": A2("a", "w", lambda n, c: ("A-data", 1)),
      "b": A2("b", "w", lambda n, c: ("B-data", 1))}
assert parallel(kb, ["a", "b"]) == {"a": "A-data", "b": "B-data"} # fan-out
sv  = A2("solver", "s", solver_backend)
cr  = A2("critic", "c", critic_backend)
assert abs(reflect(TASKS[0], sv, cr) - true_total(TASKS[0])) < 0.01   # reflection
rprint("[green]OK[/] orchestra/ package written and all four topologies re-imported + smoke-tested.")
rprint(Path(BASE, "orchestra", "core.py").read_text()[:220] + " ...")

## 8. Ten ways multi-agent systems go wrong

| # | Pitfall | Fix |
|---|---|---|
| 1 | Reaching for multi-agent when one agent would do | Start single; add an agent only to close a *measured* gap |
| 2 | "More agents = smarter" | Agents don't add IQ; they add structure (parallelism, checking, specialization) |
| 3 | Chaining independent sub-tasks | Fan them out — sequential wastes wall-clock (§5) |
| 4 | Fanning out *dependent* steps | If step 2 needs step 1's output, you must chain |
| 5 | No ground-truth to compare topologies | You can't justify the cost you can't measure (§3, §6) |
| 6 | Critic that shares the generator's blind spot | Give it an *independent* method/prompt; verification must differ from generation |
| 7 | Unbounded reflection loops | Cap `max_rounds`; a stuck critic loops forever and bills forever |
| 8 | Agents talking to everyone (mesh) | Prefer a star (supervisor-worker); N² message paths are a debugging nightmare |
| 9 | Losing information across hops | Every hand-off drops context — log every `Message`, keep payloads structured |
| 10 | Ignoring the bill | Track calls + tokens per topology (§7); orchestration is 2–5× the cost |

In [ ]:
# === Verification checklist — every claim in this lesson, re-asserted ===
checks = []
def check(name, cond):
    checks.append((name, bool(cond)))
    rprint(("[green]PASS[/]" if cond else "[red]FAIL[/]") + f"  {name}")

check("Message/Agent round-trip", echo.act(Message("u","echo","task","x")).content == "echo:x")
check("hard tasks exist to expose the bug", n_hard >= 10)
check("single-agent baseline is a real plateau (0.4-0.75)", 0.4 <= BASE_ACC <= 0.75)
check("orchestrator decomposed the goal", set(plan) == {"caching","batching","eval"})
check("every worker contributed to the brief", all(s in brief for s in plan))
check("parallel == sequential results", seq == par)
check("parallel is faster than sequential", par_time < seq_time * 0.6)
check("critic loop lifts accuracy > +0.20", MULTI_ACC > BASE_ACC + 0.20)
check("critic loop reaches >= 0.95", MULTI_ACC >= 0.95)
check("multi-agent costs more calls", m_calls > s_calls)
check("multi-agent costs more tokens", m_tok > s_tok)
check("orchestra package importable", "reflect" in dir(orchestra))
check("packaged reflect() matches ground truth", abs(reflect(TASKS[1], sv, cr) - true_total(TASKS[1])) < 0.01)

passed = sum(ok for _, ok in checks)
rprint(f"\n[bold]{passed}/{len(checks)} checks passed[/]")
assert passed == len(checks), "some checks failed"
rprint("[bold green]All verification checks passed.[/]")

## 9. Summary, homework, and what's next

### What you built today

| Topology | Shape | Reach for it when… |
|---|---|---|
| **A. Orchestrator–Worker** | star: boss → specialists | the goal decomposes into sub-tasks |
| **B. Sequential pipeline** | chain: 1 → 2 → 3 | each step depends on the last |
| **C. Parallel fan-out/in** | map → reduce | sub-tasks are independent (latency win) |
| **D. Evaluator–Optimizer** | generate → critique → revise | a verifier can catch the generator's mistakes |

The through-line: **multi-agent design is topology design.** You pick a shape based on *dependency* (chain vs. fan-out) and *quality needs* (add a critic), and you always pay for it in tokens and latency — so you measure first. Today the critic loop took the invoice solver from **~50% to ~100%**, and you packaged all four topologies into a reusable `orchestra/` module.

### 💡 Homework (pick 1–2)
1. **Wire a real critic.** Replace `critic_backend` with a Claude call (gated on `HAVE_API_KEY`) whose prompt is *"recompute this invoice independently and flag any disagreement."* Does the LLM critic recover the same lift the exact-math critic did?
2. **Add a router.** Write a `route(task)` that sends easy invoices straight to the solver (skip the critic to save tokens) and hard ones through the critic loop. Measure the token savings at equal accuracy.
3. **Break the star.** Turn the supervisor-worker demo into a *sequential* research pipeline (researcher → summarizer → editor) using `orchestra.sequential`, and compare the brief quality/latency.
4. **Stress the loop.** Give the critic a 10% false-positive rate (flags correct answers). What `max_rounds` and tolerance keep accuracy high without runaway cost? (This previews reliability, Lesson 81.)
5. **Instrument it.** Emit one agent-obs span (Phase 8) per agent hop, then query "show every task where the critic fired." You now have observability *over a multi-agent system*.

### Next up — Lesson 78: Shared state & the blackboard
Passing messages point-to-point works for four agents; at twenty, the message paths explode. Next lesson introduces the **blackboard** pattern — a shared, structured workspace agents read and write instead of messaging each other directly — plus how to keep that shared state consistent when agents run in parallel. That's the bridge from "a few wired agents" to "a coordinated system," and it sets up routing (L79) and planning (L80).

*You now own the four fundamental shapes. Phase 9 is about combining them into systems that stay debuggable as they grow.*